# Model Training Pipeline

## Supply Chain Late Delivery Prediction

---

### Modeling Strategy

This notebook implements a rigorous, production-ready modeling pipeline with the following phases:

| Phase | Description | Purpose |
|-------|-------------|----------|
| 1. Data Loading | Load and split data with stratification | Ensure reproducibility |
| 2. Feature Selection | Automated, leakage-aware selection | Prevent data leakage |
| 3. Baseline | Logistic Regression | Establish minimum benchmark |
| 4. Candidate Models | LightGBM (core), XGBoost, CatBoost | State-of-the-art performance |
| 5. Hyperparameter Tuning | Optuna with parallelization | Optimize performance |
| 6. Threshold Optimization | Optimize for F1/business metric | Balance precision/recall |
| 7. Ensembling | CV-based stacking | Combine model strengths |
| 8. Final Selection | Compare all candidates | Select production model |

### Key Principles

- **No data leakage**: Strict separation of train/val/test
- **Reproducibility**: Fixed random seeds throughout
- **Parallelization**: Multi-core training where possible
- **Explicit rationale**: Document all decisions

---

In [ ]:
# ============================================================
# SETUP & IMPORTS
# ============================================================
import sys
import warnings
import time
import os
from datetime import datetime
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.append('..')

# Core libraries
import pandas as pd
import numpy as np
import joblib

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"

# Sklearn
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, 
    RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, precision_recall_curve, 
    confusion_matrix, classification_report, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)

# Boosting libraries
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Optuna for hyperparameter tuning
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Parallel processing
from joblib import Parallel, delayed

# Configuration
RANDOM_STATE = 42
N_JOBS = -1  # Use all cores
TEST_SIZE = 0.2
VAL_SIZE = 0.1

print("Libraries loaded successfully")
print(f"Parallelization: {os.cpu_count()} CPU cores available")

---

## Phase 1: Data Loading & Preprocessing

**Objective**: Load data and create stratified train/validation/test splits.

**Key Points**:
- Stratified split preserves class distribution
- Validation set used for hyperparameter tuning and threshold optimization
- Test set held out until final evaluation

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================
from src.data.preprocess import load_or_preprocess
from src.features.build_features import build_features_pipeline

# Load and preprocess
df = load_or_preprocess()
X_raw, y = build_features_pipeline(df)

print(f"\nRaw feature matrix: {X_raw.shape[0]:,} samples x {X_raw.shape[1]} features")
print(f"Target: {y.sum():,} late ({y.mean()*100:.1f}%) / {(y==0).sum():,} on-time ({(1-y.mean())*100:.1f}%)")

In [ ]:
# ============================================================
# TRAIN/VAL/TEST SPLIT
# ============================================================

# First split: separate test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_raw, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# Second split: separate validation from training
val_size_adjusted = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=val_size_adjusted,
    random_state=RANDOM_STATE,
    stratify=y_train_val
)

print("DATA SPLIT SUMMARY")
print("=" * 50)
print(f"Train set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X_raw)*100:.0f}%)")
print(f"Val set:   {X_val.shape[0]:,} samples ({X_val.shape[0]/len(X_raw)*100:.0f}%)")
print(f"Test set:  {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X_raw)*100:.0f}%)")
print(f"\nClass distribution preserved:")
print(f"   Train late rate: {y_train.mean()*100:.1f}%")
print(f"   Val late rate:   {y_val.mean()*100:.1f}%")
print(f"   Test late rate:  {y_test.mean()*100:.1f}%")

---

## Phase 2: Automated Feature Selection

**Objective**: Apply leakage-aware, automated feature selection.

**Selection Criteria**:
1. **Leakage Prevention**: Exclude post-outcome features
2. **Production Availability**: Only features available at prediction time
3. **Statistical Filters**: Remove low-variance and high-correlation features
4. **Importance-Based**: Keep features with significant predictive power

In [ ]:
# ============================================================
# FEATURE SELECTION
# ============================================================
from src.features.feature_selector import FeatureSelector

# Initialize selector
selector = FeatureSelector(
    variance_threshold=0.01,
    correlation_threshold=0.95,
    missing_threshold=0.5,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS
)

# Run selection on training data only (to prevent leakage)
X_train_selected, selection_report = selector.fit_transform(X_train, y_train)

# Apply same selection to val and test
selected_features = selector.get_selected_features()
X_val_selected = X_val[selected_features]
X_test_selected = X_test[selected_features]

print(f"\nSelected {len(selected_features)} features for modeling")

In [ ]:
# Display feature selection report
print("FEATURE SELECTION REPORT")
print("=" * 80)

# Show dropped features
dropped_df = selection_report[selection_report['selection_status'] == 'dropped'].head(10)
if len(dropped_df) > 0:
    print("\nDropped Features (sample):")
    print(dropped_df[['feature_name', 'reason_for_decision']].to_string(index=False))

# Show selected features
selected_df = selection_report[selection_report['selection_status'] == 'selected'].head(15)
print("\nSelected Features (top 15 by importance):")
print(selected_df[['feature_name', 'importance_score', 'reason_for_decision']].to_string(index=False))

In [ ]:
# Visualize feature importances
importance_df = selector.get_feature_importance_ranking()

fig = go.Figure()
fig.add_trace(go.Bar(
    y=importance_df['feature'].head(15)[::-1],
    x=importance_df['importance'].head(15)[::-1],
    orientation='h',
    marker_color='#3498db',
    text=[f"{v:.3f}" for v in importance_df['importance'].head(15)[::-1]],
    textposition='outside'
))

fig.update_layout(
    title='<b>Top 15 Features by Importance</b>',
    xaxis_title='Relative Importance',
    height=500,
    showlegend=False
)
fig.show()

---

## Phase 3: Baseline Model

**Objective**: Establish minimum performance benchmark with Logistic Regression.

**Why Logistic Regression**:
- Simple, interpretable baseline
- Fast training and inference
- Any model should beat this baseline

In [ ]:
# ============================================================
# PHASE 3: BASELINE MODEL (Logistic Regression)
# ============================================================

# Results storage
results = {}

print("PHASE 3: BASELINE MODEL")
print("=" * 60)

# Train Logistic Regression
start_time = time.time()
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=N_JOBS,
    solver='lbfgs'
)
baseline_model.fit(X_train_selected, y_train)
baseline_time = time.time() - start_time

# Evaluate on validation set
y_val_pred_baseline = baseline_model.predict(X_val_selected)
y_val_proba_baseline = baseline_model.predict_proba(X_val_selected)[:, 1]

results['Baseline (Logistic)'] = {
    'model': baseline_model,
    'val_accuracy': accuracy_score(y_val, y_val_pred_baseline),
    'val_f1': f1_score(y_val, y_val_pred_baseline, average='weighted'),
    'val_precision': precision_score(y_val, y_val_pred_baseline, average='weighted'),
    'val_recall': recall_score(y_val, y_val_pred_baseline, average='weighted'),
    'val_roc_auc': roc_auc_score(y_val, y_val_proba_baseline),
    'train_time': baseline_time,
    'y_proba': y_val_proba_baseline,
    'threshold': 0.5,
    'tuned': False
}

print(f"\nLogistic Regression Baseline (trained in {baseline_time:.2f}s):")
print(f"   Validation Accuracy: {results['Baseline (Logistic)']['val_accuracy']:.4f}")
print(f"   Validation F1 Score: {results['Baseline (Logistic)']['val_f1']:.4f}")
print(f"   Validation ROC-AUC:  {results['Baseline (Logistic)']['val_roc_auc']:.4f}")
print("\n" + "="*60)
print("BASELINE ESTABLISHED - All models must beat these metrics")
print("="*60)

---

## Phase 4: Candidate Models

**Objective**: Train candidate models with sensible defaults.

### Model Selection Rationale

| Model | Why Include | Strengths |
|-------|-------------|----------|
| **LightGBM (Core)** | Fast, memory-efficient, handles categoricals | Best for tabular data, excellent parallelization |
| **XGBoost** | Industry standard, robust | Strong regularization, proven track record |
| **CatBoost** | Minimal tuning needed | Native categorical handling, robust to overfitting |
| **Random Forest** | Ensemble baseline | Less prone to overfitting, interpretable |

**LightGBM as Core Model**:
- Fastest training among gradient boosting methods
- Leaf-wise growth provides better accuracy
- Native handling of categorical features
- Excellent parallelization with `num_threads`

In [ ]:
# ============================================================
# PHASE 4: CANDIDATE MODELS (UNTUNED)
# ============================================================

print("PHASE 4: CANDIDATE MODELS (Default Parameters)")
print("=" * 60)

# Define candidate models with sensible defaults
candidates = {
    'LightGBM': LGBMClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=-1,
        force_col_wise=True
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        eval_metric='logloss',
        use_label_encoder=False
    ),
    'CatBoost': CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        auto_class_weights='Balanced',
        verbose=False,
        thread_count=N_JOBS if N_JOBS > 0 else -1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    )
}

def train_and_evaluate(name, model, X_tr, y_tr, X_v, y_v):
    """Train model and return evaluation metrics."""
    start = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - start
    
    y_pred = model.predict(X_v)
    y_proba = model.predict_proba(X_v)[:, 1]
    
    return {
        'name': name,
        'model': model,
        'val_accuracy': accuracy_score(y_v, y_pred),
        'val_f1': f1_score(y_v, y_pred, average='weighted'),
        'val_precision': precision_score(y_v, y_pred, average='weighted'),
        'val_recall': recall_score(y_v, y_pred, average='weighted'),
        'val_roc_auc': roc_auc_score(y_v, y_proba),
        'train_time': train_time,
        'y_proba': y_proba,
        'threshold': 0.5,
        'tuned': False
    }

# Train all candidates (could be parallelized but kept sequential for stability)
for name, model in candidates.items():
    print(f"\nTraining {name}...")
    result = train_and_evaluate(name, model, X_train_selected, y_train, X_val_selected, y_val)
    results[name] = result
    print(f"   Val F1: {result['val_f1']:.4f}, Val ROC-AUC: {result['val_roc_auc']:.4f} ({result['train_time']:.2f}s)")

print("\n" + "="*60)
print("UNTUNED CANDIDATE MODELS TRAINED")
print("="*60)

---

## Phase 5: Hyperparameter Tuning with Optuna

**Objective**: Optimize hyperparameters using Bayesian optimization.

**Why Optuna**:
- Efficient Bayesian search (TPE sampler)
- Automatic pruning of unpromising trials
- Native parallelization support
- Better than grid/random search for complex spaces

**Focus**: Tune LightGBM (core) and XGBoost (runner-up) extensively.

In [ ]:
# ============================================================
# PHASE 5: HYPERPARAMETER TUNING (Optuna)
# ============================================================

print("PHASE 5: HYPERPARAMETER TUNING (Optuna)")
print("=" * 60)

N_TRIALS = 50  # Number of Optuna trials per model
CV_FOLDS = 3   # Cross-validation folds for tuning

def create_lightgbm_objective(X_tr, y_tr):
    """Create Optuna objective for LightGBM."""
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 100),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'class_weight': 'balanced',
            'random_state': RANDOM_STATE,
            'n_jobs': 1,  # Use 1 job per model in parallel tuning
            'verbose': -1,
            'force_col_wise': True
        }
        
        model = LGBMClassifier(**params)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='f1_weighted', n_jobs=1)
        return scores.mean()
    
    return objective

def create_xgboost_objective(X_tr, y_tr):
    """Create Optuna objective for XGBoost."""
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': RANDOM_STATE,
            'n_jobs': 1,
            'eval_metric': 'logloss',
            'use_label_encoder': False
        }
        
        model = XGBClassifier(**params)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='f1_weighted', n_jobs=1)
        return scores.mean()
    
    return objective

# Tune LightGBM
print("\nTuning LightGBM...")
start_time = time.time()
lgbm_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)
lgbm_study.optimize(
    create_lightgbm_objective(X_train_selected, y_train),
    n_trials=N_TRIALS,
    n_jobs=N_JOBS,
    show_progress_bar=True
)
lgbm_tune_time = time.time() - start_time

print(f"   Best CV F1: {lgbm_study.best_value:.4f}")
print(f"   Tuning time: {lgbm_tune_time:.1f}s")

# Tune XGBoost
print("\nTuning XGBoost...")
start_time = time.time()
xgb_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE)
)
xgb_study.optimize(
    create_xgboost_objective(X_train_selected, y_train),
    n_trials=N_TRIALS,
    n_jobs=N_JOBS,
    show_progress_bar=True
)
xgb_tune_time = time.time() - start_time

print(f"   Best CV F1: {xgb_study.best_value:.4f}")
print(f"   Tuning time: {xgb_tune_time:.1f}s")

In [ ]:
# Train tuned models on full training set
print("\nTraining tuned models on full training set...")

# LightGBM Tuned
lgbm_tuned = LGBMClassifier(
    **lgbm_study.best_params,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=-1,
    force_col_wise=True
)
start = time.time()
lgbm_tuned.fit(X_train_selected, y_train)
lgbm_train_time = time.time() - start

y_val_pred_lgbm = lgbm_tuned.predict(X_val_selected)
y_val_proba_lgbm = lgbm_tuned.predict_proba(X_val_selected)[:, 1]

results['LightGBM (Tuned)'] = {
    'model': lgbm_tuned,
    'val_accuracy': accuracy_score(y_val, y_val_pred_lgbm),
    'val_f1': f1_score(y_val, y_val_pred_lgbm, average='weighted'),
    'val_precision': precision_score(y_val, y_val_pred_lgbm, average='weighted'),
    'val_recall': recall_score(y_val, y_val_pred_lgbm, average='weighted'),
    'val_roc_auc': roc_auc_score(y_val, y_val_proba_lgbm),
    'train_time': lgbm_train_time,
    'y_proba': y_val_proba_lgbm,
    'threshold': 0.5,
    'tuned': True,
    'best_params': lgbm_study.best_params
}

print(f"   LightGBM (Tuned): Val F1={results['LightGBM (Tuned)']['val_f1']:.4f}")

# XGBoost Tuned
xgb_tuned = XGBClassifier(
    **xgb_study.best_params,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    eval_metric='logloss',
    use_label_encoder=False
)
start = time.time()
xgb_tuned.fit(X_train_selected, y_train)
xgb_train_time = time.time() - start

y_val_pred_xgb = xgb_tuned.predict(X_val_selected)
y_val_proba_xgb = xgb_tuned.predict_proba(X_val_selected)[:, 1]

results['XGBoost (Tuned)'] = {
    'model': xgb_tuned,
    'val_accuracy': accuracy_score(y_val, y_val_pred_xgb),
    'val_f1': f1_score(y_val, y_val_pred_xgb, average='weighted'),
    'val_precision': precision_score(y_val, y_val_pred_xgb, average='weighted'),
    'val_recall': recall_score(y_val, y_val_pred_xgb, average='weighted'),
    'val_roc_auc': roc_auc_score(y_val, y_val_proba_xgb),
    'train_time': xgb_train_time,
    'y_proba': y_val_proba_xgb,
    'threshold': 0.5,
    'tuned': True,
    'best_params': xgb_study.best_params
}

print(f"   XGBoost (Tuned): Val F1={results['XGBoost (Tuned)']['val_f1']:.4f}")

print("\n" + "="*60)
print("HYPERPARAMETER TUNING COMPLETE")
print("="*60)

---

## Phase 6: Threshold Optimization

**Objective**: Find optimal classification threshold for business needs.

**Default (0.5) may not be optimal** when:
- Classes are imbalanced
- False negatives cost more than false positives (or vice versa)
- Business requires specific precision/recall tradeoff

**Optimization Strategies**:
- **F1 Optimization**: Balance precision and recall
- **Youden's J**: Maximize TPR - FPR
- **Business Cost**: Minimize expected cost

In [ ]:
# ============================================================
# PHASE 6: THRESHOLD OPTIMIZATION
# ============================================================

print("PHASE 6: THRESHOLD OPTIMIZATION")
print("=" * 60)

def optimize_threshold(y_true, y_proba, metric='f1'):
    """
    Find optimal classification threshold.
    
    Args:
        y_true: True labels
        y_proba: Predicted probabilities
        metric: 'f1', 'youden', or 'f1_binary'
        
    Returns:
        best_threshold, best_score, threshold_scores
    """
    thresholds = np.arange(0.1, 0.9, 0.01)
    scores = []
    
    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        
        if metric == 'f1':
            score = f1_score(y_true, y_pred, average='weighted')
        elif metric == 'f1_binary':
            score = f1_score(y_true, y_pred, average='binary')
        elif metric == 'youden':
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
            score = tpr - fpr
        else:
            score = f1_score(y_true, y_pred, average='weighted')
            
        scores.append(score)
    
    best_idx = np.argmax(scores)
    return thresholds[best_idx], scores[best_idx], list(zip(thresholds, scores))

# Optimize threshold for top models
models_to_optimize = ['LightGBM (Tuned)', 'XGBoost (Tuned)', 'CatBoost']

threshold_results = {}
for model_name in models_to_optimize:
    if model_name in results:
        y_proba = results[model_name]['y_proba']
        best_thresh, best_score, thresh_scores = optimize_threshold(y_val, y_proba, metric='f1')
        
        # Update results with optimized threshold
        y_pred_opt = (y_proba >= best_thresh).astype(int)
        
        results[f'{model_name} (Opt Thresh)'] = {
            'model': results[model_name]['model'],
            'val_accuracy': accuracy_score(y_val, y_pred_opt),
            'val_f1': f1_score(y_val, y_pred_opt, average='weighted'),
            'val_precision': precision_score(y_val, y_pred_opt, average='weighted'),
            'val_recall': recall_score(y_val, y_pred_opt, average='weighted'),
            'val_roc_auc': results[model_name]['val_roc_auc'],
            'train_time': results[model_name]['train_time'],
            'y_proba': y_proba,
            'threshold': best_thresh,
            'tuned': results[model_name].get('tuned', False)
        }
        
        threshold_results[model_name] = {'threshold': best_thresh, 'scores': thresh_scores}
        
        print(f"\n{model_name}:")
        print(f"   Default threshold (0.5): F1={results[model_name]['val_f1']:.4f}")
        print(f"   Optimal threshold ({best_thresh:.2f}): F1={best_score:.4f}")
        improvement = best_score - results[model_name]['val_f1']
        print(f"   Improvement: {'+' if improvement > 0 else ''}{improvement:.4f}")

In [ ]:
# Visualize threshold optimization
fig = make_subplots(rows=1, cols=len(threshold_results), 
                    subplot_titles=list(threshold_results.keys()))

for i, (model_name, data) in enumerate(threshold_results.items(), 1):
    thresholds, scores = zip(*data['scores'])
    optimal_thresh = data['threshold']
    
    fig.add_trace(
        go.Scatter(x=list(thresholds), y=list(scores), mode='lines', 
                   name=model_name, line=dict(width=2)),
        row=1, col=i
    )
    
    # Mark optimal point
    optimal_score = scores[list(thresholds).index(optimal_thresh)]
    fig.add_trace(
        go.Scatter(x=[optimal_thresh], y=[optimal_score], mode='markers',
                   marker=dict(size=12, color='red', symbol='star'),
                   name=f'Optimal ({optimal_thresh:.2f})', showlegend=False),
        row=1, col=i
    )

fig.update_layout(
    title='<b>Threshold Optimization: F1 Score vs Threshold</b>',
    height=400,
    showlegend=True
)
fig.update_xaxes(title_text='Threshold')
fig.update_yaxes(title_text='F1 Score')
fig.show()

print("\n" + "="*60)
print("THRESHOLD OPTIMIZATION COMPLETE")
print("="*60)

---

## Phase 7: Ensembling

**Objective**: Combine top models to leverage their complementary strengths.

**Ensemble Strategies**:

| Strategy | Description | When to Use |
|----------|-------------|-------------|
| **Voting** | Average/vote predictions | Quick, robust |
| **Stacking** | Meta-learner on base predictions | Best performance |
| **Blending** | Weighted average of probabilities | Simple, interpretable |

**CV-Based Stacking**: Use cross-validation to generate meta-features to avoid data leakage.

In [ ]:
# ============================================================
# PHASE 7: ENSEMBLING
# ============================================================

print("PHASE 7: ENSEMBLE MODELING")
print("=" * 60)

# Get best tuned models for ensembling
base_estimators = [
    ('lgbm', LGBMClassifier(
        **lgbm_study.best_params,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=-1,
        force_col_wise=True
    )),
    ('xgb', XGBClassifier(
        **xgb_study.best_params,
        random_state=RANDOM_STATE,
        n_jobs=1,
        eval_metric='logloss',
        use_label_encoder=False
    )),
    ('catboost', CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        auto_class_weights='Balanced',
        verbose=False,
        thread_count=1
    ))
]

# 1. Voting Ensemble
print("\n1. Training Voting Ensemble...")
start = time.time()
voting_clf = VotingClassifier(
    estimators=base_estimators,
    voting='soft',
    n_jobs=N_JOBS
)
voting_clf.fit(X_train_selected, y_train)
voting_time = time.time() - start

y_val_pred_voting = voting_clf.predict(X_val_selected)
y_val_proba_voting = voting_clf.predict_proba(X_val_selected)[:, 1]

results['Voting Ensemble'] = {
    'model': voting_clf,
    'val_accuracy': accuracy_score(y_val, y_val_pred_voting),
    'val_f1': f1_score(y_val, y_val_pred_voting, average='weighted'),
    'val_precision': precision_score(y_val, y_val_pred_voting, average='weighted'),
    'val_recall': recall_score(y_val, y_val_pred_voting, average='weighted'),
    'val_roc_auc': roc_auc_score(y_val, y_val_proba_voting),
    'train_time': voting_time,
    'y_proba': y_val_proba_voting,
    'threshold': 0.5,
    'tuned': True
}
print(f"   Val F1: {results['Voting Ensemble']['val_f1']:.4f} ({voting_time:.2f}s)")

# 2. Stacking Ensemble (CV-based to avoid leakage)
print("\n2. Training Stacking Ensemble (CV-based)...")
start = time.time()
stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5,  # 5-fold CV for meta-learner training
    n_jobs=N_JOBS,
    passthrough=False  # Only use base model predictions
)
stacking_clf.fit(X_train_selected, y_train)
stacking_time = time.time() - start

y_val_pred_stacking = stacking_clf.predict(X_val_selected)
y_val_proba_stacking = stacking_clf.predict_proba(X_val_selected)[:, 1]

results['Stacking Ensemble'] = {
    'model': stacking_clf,
    'val_accuracy': accuracy_score(y_val, y_val_pred_stacking),
    'val_f1': f1_score(y_val, y_val_pred_stacking, average='weighted'),
    'val_precision': precision_score(y_val, y_val_pred_stacking, average='weighted'),
    'val_recall': recall_score(y_val, y_val_pred_stacking, average='weighted'),
    'val_roc_auc': roc_auc_score(y_val, y_val_proba_stacking),
    'train_time': stacking_time,
    'y_proba': y_val_proba_stacking,
    'threshold': 0.5,
    'tuned': True
}
print(f"   Val F1: {results['Stacking Ensemble']['val_f1']:.4f} ({stacking_time:.2f}s)")

print("\n" + "="*60)
print("ENSEMBLE MODELS TRAINED")
print("="*60)

---

## Phase 8: Final Model Selection

**Objective**: Select the best model for production based on multiple criteria.

**Selection Criteria**:
1. **Validation Performance**: Primary metric (F1, ROC-AUC)
2. **Stability**: Consistent performance across folds
3. **Inference Speed**: Production latency requirements
4. **Interpretability**: Stakeholder requirements

**Final Evaluation**: On held-out test set (only once!).

In [ ]:
# ============================================================
# MODEL COMPARISON
# ============================================================

print("PHASE 8: FINAL MODEL SELECTION")
print("=" * 60)

# Create comparison DataFrame
comparison_data = []
for name, res in results.items():
    comparison_data.append({
        'Model': name,
        'Val Accuracy': res['val_accuracy'],
        'Val F1': res['val_f1'],
        'Val Precision': res['val_precision'],
        'Val Recall': res['val_recall'],
        'Val ROC-AUC': res['val_roc_auc'],
        'Threshold': res['threshold'],
        'Train Time (s)': res['train_time'],
        'Tuned': res.get('tuned', False)
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('Val F1', ascending=False)

print("\nMODEL COMPARISON (sorted by Val F1):")
print(comparison_df[['Model', 'Val F1', 'Val ROC-AUC', 'Val Precision', 'Val Recall', 'Threshold']].to_string(index=False))

In [ ]:
# Visualize model comparison
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>F1 Score by Model</b>', '<b>ROC-AUC by Model</b>')
)

# Sort for visualization
sorted_df = comparison_df.sort_values('Val F1', ascending=True).tail(10)

# F1 Score
colors = ['#3498db' if not t else '#e74c3c' for t in sorted_df['Tuned']]
fig.add_trace(
    go.Bar(
        y=sorted_df['Model'],
        x=sorted_df['Val F1'],
        orientation='h',
        marker_color=colors,
        text=[f"{v:.4f}" for v in sorted_df['Val F1']],
        textposition='outside',
        name='F1'
    ),
    row=1, col=1
)

# ROC-AUC
fig.add_trace(
    go.Bar(
        y=sorted_df['Model'],
        x=sorted_df['Val ROC-AUC'],
        orientation='h',
        marker_color=colors,
        text=[f"{v:.4f}" for v in sorted_df['Val ROC-AUC']],
        textposition='outside',
        name='ROC-AUC'
    ),
    row=1, col=2
)

fig.update_layout(
    height=500,
    title='<b>Model Performance Comparison</b><br><sup>Blue=Untuned, Red=Tuned</sup>',
    showlegend=False
)
fig.update_xaxes(range=[0.5, 1.0])
fig.show()

In [ ]:
# Select best model
best_model_name = comparison_df.iloc[0]['Model']
best_model_result = results[best_model_name]

print("\n" + "="*60)
print(f"SELECTED MODEL: {best_model_name}")
print("="*60)
print(f"\nValidation Performance:")
print(f"   F1 Score:  {best_model_result['val_f1']:.4f}")
print(f"   ROC-AUC:   {best_model_result['val_roc_auc']:.4f}")
print(f"   Precision: {best_model_result['val_precision']:.4f}")
print(f"   Recall:    {best_model_result['val_recall']:.4f}")
print(f"   Threshold: {best_model_result['threshold']:.2f}")

# Baseline comparison
baseline_f1 = results['Baseline (Logistic)']['val_f1']
improvement = best_model_result['val_f1'] - baseline_f1
print(f"\nImprovement over baseline: +{improvement:.4f} F1 ({improvement/baseline_f1*100:.1f}%)")

In [ ]:
# ============================================================
# FINAL TEST SET EVALUATION
# ============================================================

print("\n" + "="*60)
print("FINAL TEST SET EVALUATION")
print("="*60)

best_model = best_model_result['model']
best_threshold = best_model_result['threshold']

# Predict on test set
y_test_proba = best_model.predict_proba(X_test_selected)[:, 1]
y_test_pred = (y_test_proba >= best_threshold).astype(int)

# Calculate metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred, average='weighted')
test_precision = precision_score(y_test, y_test_pred, average='weighted')
test_recall = recall_score(y_test, y_test_pred, average='weighted')
test_roc_auc = roc_auc_score(y_test, y_test_proba)

print(f"\n{best_model_name} - Test Set Results:")
print(f"   Accuracy:  {test_accuracy:.4f}")
print(f"   F1 Score:  {test_f1:.4f}")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   ROC-AUC:   {test_roc_auc:.4f}")

# Classification Report
print(f"\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=['On-Time', 'Late']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix:")
print(f"   True Positives (Late caught):      {tp:,}")
print(f"   True Negatives (On-time correct):  {tn:,}")
print(f"   False Positives (False alarms):    {fp:,}")
print(f"   False Negatives (Missed late):     {fn:,}")

In [ ]:
# Visualize test results
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "heatmap"}, {"type": "scatter"}, {"type": "scatter"}]],
    subplot_titles=('<b>Confusion Matrix</b>', '<b>ROC Curve</b>', '<b>Precision-Recall Curve</b>')
)

# Confusion Matrix
fig.add_trace(
    go.Heatmap(
        z=cm,
        x=['Pred: On-Time', 'Pred: Late'],
        y=['Actual: On-Time', 'Actual: Late'],
        colorscale='Blues',
        text=cm,
        texttemplate='%{text:,}',
        textfont={'size': 14},
        showscale=False
    ),
    row=1, col=1
)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
fig.add_trace(
    go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC (AUC={test_roc_auc:.3f})',
               line=dict(color='#e74c3c', width=2)),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random',
               line=dict(color='gray', dash='dash')),
    row=1, col=2
)

# PR Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_proba)
fig.add_trace(
    go.Scatter(x=recall_curve, y=precision_curve, mode='lines', name='PR Curve',
               line=dict(color='#2ecc71', width=2)),
    row=1, col=3
)

fig.update_layout(
    height=400,
    title=f'<b>{best_model_name} - Test Set Performance</b>',
    showlegend=True
)
fig.update_xaxes(title_text='False Positive Rate', row=1, col=2)
fig.update_yaxes(title_text='True Positive Rate', row=1, col=2)
fig.update_xaxes(title_text='Recall', row=1, col=3)
fig.update_yaxes(title_text='Precision', row=1, col=3)
fig.show()

In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

model_dir = Path('../models')
model_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# Save best model
model_path = model_dir / f'best_model_{timestamp}.pkl'
joblib.dump(best_model, model_path)
print(f"Model saved to: {model_path}")

# Save feature selector
selector_path = model_dir / f'feature_selector_{timestamp}.pkl'
joblib.dump(selector, selector_path)
print(f"Feature selector saved to: {selector_path}")

# Save results summary
results_summary = {
    'best_model_name': best_model_name,
    'best_threshold': best_threshold,
    'test_f1': test_f1,
    'test_roc_auc': test_roc_auc,
    'test_accuracy': test_accuracy,
    'selected_features': selected_features,
    'comparison_df': comparison_df.to_dict()
}
results_path = model_dir / f'training_results_{timestamp}.pkl'
joblib.dump(results_summary, results_path)
print(f"Results saved to: {results_path}")

# Save feature selection report
report_path = model_dir / f'feature_selection_report_{timestamp}.csv'
selection_report.to_csv(report_path, index=False)
print(f"Feature selection report saved to: {report_path}")

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("MODEL TRAINING PIPELINE COMPLETE")
print("=" * 80)

print(f"""
PIPELINE SUMMARY
{"="*60}

1. DATA
   - Total samples: {len(X_raw):,}
   - Train/Val/Test split: {len(X_train):,}/{len(X_val):,}/{len(X_test):,}
   - Features (raw): {X_raw.shape[1]}
   - Features (selected): {len(selected_features)}

2. MODELS TRAINED
   - Baseline: Logistic Regression (F1={results['Baseline (Logistic)']['val_f1']:.4f})
   - Candidates: LightGBM, XGBoost, CatBoost, Random Forest
   - Tuned: LightGBM, XGBoost (Optuna, {N_TRIALS} trials each)
   - Ensembles: Voting, Stacking (5-fold CV)

3. BEST MODEL: {best_model_name}
   - Test F1 Score:  {test_f1:.4f}
   - Test ROC-AUC:   {test_roc_auc:.4f}
   - Test Precision: {test_precision:.4f}
   - Test Recall:    {test_recall:.4f}
   - Threshold:      {best_threshold:.2f}

4. IMPROVEMENT OVER BASELINE
   - F1 improvement: +{improvement:.4f} ({improvement/baseline_f1*100:.1f}%)

5. BUSINESS METRICS
   - Late deliveries caught: {tp:,} out of {tp+fn:,} ({tp/(tp+fn)*100:.1f}%)
   - False alarms: {fp:,} ({fp/(fp+tn)*100:.1f}% of on-time orders)

{"="*60}
Model saved to: {model_path}
Next: Run 05_business_impact.ipynb for SHAP analysis
""")